# 🛠️ NB4 · Workshop: Build a Pipeline a Stranger Can Re-run · Taller: un pipeline que un extraño pueda repetir
**Research Methods & Scientific Integrity in AI · Session 5 · UNMSM Deep Tech Doctorate · Dr. Loveleen Gaur**  
*Métodos de Investigación e Integridad Científica en IA · Sesión 5*


**EN:** Hands-on workshop (about 45 min). Four steps. At the end you have a small GitHub project that anyone can re-run and get **your** number.  
**ES:** Taller práctico (unos 45 min). Cuatro pasos. Al final tendrás un pequeño proyecto en GitHub que cualquiera puede ejecutar y obtener **tu** número.

| Step · Paso | What · Qué | Tool · Herramienta | Min |
|---|---|---|---|
| 0 | Setup · Preparación | pip, git | 5 |
| 1 | Save your code history · Guarda el historial del código | Git | 8 |
| 2 | Same seed, same result, no leakage · Misma semilla, mismo resultado, sin fuga | `train.py` | 12 |
| 3 | Record every run · Registra cada ejecución | MLflow | 10 |
| 4 | Recipe for a stranger · Receta para un extraño | `requirements.txt`, README, GitHub | 10 |

**🔒 Safety · Seguridad:** never type your GitHub token inside a code cell. Step 4 asks for it in a hidden box and does not save it.  
*Nunca escribas tu token de GitHub dentro de una celda. El paso 4 lo pide en una caja oculta y no lo guarda.*

> **🇪🇸 Traducción al español**
>
> # 🛠️ NB4 · Taller: construye un pipeline que un extraño pueda volver a ejecutar
> **Métodos de Investigación e Integridad Científica en IA · Sesión 5 · Doctorado en Deep Tech UNMSM · Dra. Loveleen Gaur**
>
> Taller práctico (unos 45 min). Cuatro pasos. Al final tendrás un pequeño proyecto en GitHub que cualquiera puede ejecutar para obtener **tu** número.
>
> | Paso | Qué | Herramienta | Min |
> |---|---|---|---|
> | 0 | Preparación | pip, git | 5 |
> | 1 | Guarda el historial de tu código | Git | 8 |
> | 2 | Misma semilla, mismo resultado, sin fuga de datos | `train.py` | 12 |
> | 3 | Registra cada ejecución | MLflow | 10 |
> | 4 | Receta para un extraño | `requirements.txt`, README, GitHub | 10 |
>
> **🔒 Seguridad:** nunca escribas tu token de GitHub dentro de una celda de código. El paso 4 lo pide en una caja oculta y no lo guarda.

## Step 0 · Setup · Preparación
**EN:** Run this during the break. Change the name and email to yours.  
**ES:** Ejecútalo durante el descanso. Cambia el nombre y correo por los tuyos.

> **🇪🇸 Traducción al español**
>
> ## Paso 0 · Preparación
> Ejecútalo durante el descanso. Cambia el nombre y el correo por los tuyos.

In [2]:
# Instala MLflow en silencio (-q)
!pip -q install mlflow
# Importa módulos del sistema operativo, del sistema y de la plataforma
import os, sys, platform
# Configura tu nombre en Git (edítalo)
!git config --global user.name  "FELIX RICHAR MENDOZA PUMACAHUA"          # EN: edit / ES: edita
# Configura tu correo en Git (edítalo)
!git config --global user.email "felix.mendozap@unmsm.edu.pe"    # EN: edit / ES: edita
# Define "main" como nombre de la rama por defecto
!git config --global init.defaultBranch main
# Confirma la versión de Python y que todo está listo
print("Python", platform.python_version(), "| ready / listo ✅")

Python 3.13.16 | ready / listo ✅


## Step 1 · Save your code history (Git) · Guarda el historial de tu código (Git)
**EN:** Git is like "track changes" for code. Each **commit** is a saved snapshot with a short ID (a *hash*). You can write that ID next to every number in your thesis.  
**ES:** Git es como el "control de cambios" del código. Cada **commit** es una foto guardada con un identificador corto (*hash*). Puedes escribir ese ID junto a cada número de tu tesis.

> **🇪🇸 Traducción al español**
>
> ## Paso 1 · Guarda el historial de tu código (Git)
> Git es como el "control de cambios" para el código. Cada **commit** es una foto guardada con un identificador corto (un *hash*). Puedes escribir ese ID junto a cada número de tu tesis.

In [3]:
# Entra a la carpeta /content (directorio de trabajo de Colab)
%cd /content
# Borra el proyecto previo (si existe) y crea las carpetas repro-lab/data y repro-lab/src
!rm -rf repro-lab && mkdir -p repro-lab/data repro-lab/src
# Entra a la carpeta del proyecto
%cd /content/repro-lab
# Inicializa un repositorio Git vacío en silencio
!git init -q

# Files Git should ignore / Archivos que Git debe ignorar
# Crea .gitignore con los archivos que Git no debe versionar (caché de Python, base de MLflow, checkpoints)
with open(".gitignore", "w") as f:
    # Escribe en .gitignore las rutas que Git debe ignorar, una por línea
    f.write("__pycache__/\nmlflow.db\nmlruns/\n.ipynb_checkpoints/\n")
# Crea un README inicial con un título y una nota de trabajo en progreso
with open("README.md", "w") as f:
    # Escribe en README.md el título y la nota de trabajo en progreso
    f.write("# Reproducible ML Pipeline · Session 5\n\nWork in progress.\n")

# Agrega ambos archivos y hace el primer commit
!git add .gitignore README.md && git commit -q -m "Start project: README and .gitignore"
# Muestra el historial de commits en una línea por commit
!git log --oneline

/content
/content/repro-lab
b31198b (HEAD -> main) Start project: README and .gitignore


**EN:** Now save the data inside the project, so everyone uses the **same file**. We also print its *fingerprint* (MD5). If the file changes, the fingerprint changes.  
**ES:** Ahora guarda los datos dentro del proyecto, para que todos usen el **mismo archivo**. También imprimimos su *huella* (MD5). Si el archivo cambia, la huella cambia.

> **🇪🇸 Traducción al español**
>
> Ahora guarda los datos dentro del proyecto, para que todos usen el **mismo archivo**. También imprimimos su *huella* (MD5). Si el archivo cambia, la huella cambia.

In [4]:
# Librería para calcular huellas (hash) MD5
import hashlib
# Carga el conjunto de datos de cáncer de mama
from sklearn.datasets import load_breast_cancer
# Guarda los datos como CSV dentro del proyecto (sin la columna de índice)
load_breast_cancer(as_frame=True).frame.to_csv("data/breast_cancer.csv", index=False)
# Calcula e imprime la huella MD5 del archivo CSV leído en binario
print("Data fingerprint / Huella de los datos (MD5):", hashlib.md5(open("data/breast_cancer.csv", "rb").read()).hexdigest())
# Agrega el CSV a Git y hace commit
!git add data/breast_cancer.csv && git commit -q -m "Add dataset (breast cancer, 569 rows)"
# Muestra el historial de commits
!git log --oneline

Data fingerprint / Huella de los datos (MD5): 5afc23b9622f8f9ffa60824f4e97d0d5
191bfc9 (HEAD -> main) Add dataset (breast cancer, 569 rows)
b31198b Start project: README and .gitignore


## Step 2 · Same seed, same result, no leakage · Misma semilla, mismo resultado, sin fuga
**EN:** This script does three good things:  
**ES:** Este script hace tres cosas buenas:
1. **Fixes the seeds** (Python and NumPy), so randomness is the same each time. · **Fija las semillas** para que el azar sea igual cada vez.
2. **Uses a `Pipeline`**, so the scaler only learns from training data (no leakage). · **Usa un `Pipeline`**, así el escalador solo aprende de los datos de entrenamiento (sin fuga).
3. **Reports mean ± SD** over 5 tests, not one lucky number. · **Reporta media ± DE** sobre 5 pruebas, no un número con suerte.

> **🇪🇸 Traducción al español**
>
> ## Paso 2 · Misma semilla, mismo resultado, sin fuga de datos
> Este script hace tres cosas buenas:
>
> 1. **Fija las semillas** (Python y NumPy), para que el azar sea igual cada vez.
> 2. **Usa un `Pipeline`**, de modo que el escalador solo aprende de los datos de entrenamiento (sin fuga de datos).
> 3. **Reporta media ± DE** sobre 5 pruebas, no un solo número con suerte.

In [5]:
%%writefile src/train.py
# Documentación (ES): Entrena y prueba un modelo de forma reproducible. Ejecutar: PYTHONHASHSEED=0 python src/train.py --seed 42
"""Train and test a model reproducibly.  Run: PYTHONHASHSEED=0 python src/train.py --seed 42"""
# Módulos: argumentos de línea de comandos, hash, JSON y semillas de Python
import argparse, hashlib, json, random
# NumPy y pandas para datos numéricos y tablas
import numpy as np, pandas as pd
# Modelo de regresión logística
from sklearn.linear_model import LogisticRegression
# Validación cruzada estratificada y su evaluación
from sklearn.model_selection import StratifiedKFold, cross_val_score
# Encadena pasos en un Pipeline
from sklearn.pipeline import make_pipeline
# Escalador que estandariza las variables
from sklearn.preprocessing import StandardScaler

# Fija las semillas de las librerías de azar
def set_seed(seed):
    # Semilla de Python
    random.seed(seed)        # Python
    # Semilla de NumPy
    np.random.seed(seed)     # NumPy  (with PyTorch, also: torch.manual_seed(seed))

# Función principal: entrena y evalúa con una semilla y una ruta de datos
def main(seed=42, data_path="data/breast_cancer.csv"):
    # Fija las semillas antes de cualquier proceso aleatorio
    set_seed(seed)
    # Lee el CSV de datos
    df = pd.read_csv(data_path)
    # Separa variables (X) y etiqueta "target" (y)
    X, y = df.drop(columns="target").values, df["target"].values
    # Pipeline: el escalador aprende solo de cada partición de entrenamiento
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))   # no leakage / sin fuga
    # Validación cruzada de 5 particiones, barajada con la semilla
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    # Exactitud de cada una de las 5 particiones
    scores = cross_val_score(model, X, y, cv=cv)
    # Devuelve semilla, huella MD5 de los datos y media ± DE de la exactitud
    return {"seed": seed,
            # Clave data_md5: huella MD5 del archivo de datos
            "data_md5": hashlib.md5(open(data_path, "rb").read()).hexdigest(),
            # Clave accuracy_mean: exactitud media redondeada a 5 decimales
            "accuracy_mean": round(float(scores.mean()), 5),
            # Clave accuracy_sd: desviación estándar redondeada a 5 decimales; cierra el diccionario
            "accuracy_sd": round(float(scores.std(ddof=1)), 5)}

# Solo se ejecuta cuando el archivo se lanza como script
if __name__ == "__main__":
    # Define el argumento --seed (entero, por defecto 42) e imprime el resultado como JSON
    ap = argparse.ArgumentParser(); ap.add_argument("--seed", type=int, default=42)
    # Lee la semilla indicada en la línea de comandos, ejecuta main e imprime el resultado en JSON
    print(json.dumps(main(ap.parse_args().seed)))

Writing src/train.py


**✅ The test · La prueba:** run it twice with the same seed. **The two lines must be identical.**  
*Ejecútalo dos veces con la misma semilla. **Las dos líneas deben ser idénticas.***

💡 `PYTHONHASHSEED=0` is written **before** `python` on purpose: this seed only works if it is set when Python starts.  
*`PYTHONHASHSEED=0` va **antes** de `python` a propósito: esta semilla solo funciona si se fija cuando Python arranca.*

> **🇪🇸 Traducción al español**
>
> **✅ La prueba:** ejecútalo dos veces con la misma semilla. **Las dos líneas deben ser idénticas.**
>
> 💡 `PYTHONHASHSEED=0` se escribe **antes** de `python` a propósito: esta semilla solo funciona si se fija cuando Python arranca.

In [6]:
# Primera ejecución con semilla 42 (PYTHONHASHSEED=0 se fija al arrancar Python)
!PYTHONHASHSEED=0 python src/train.py --seed 42
# Segunda ejecución idéntica: la salida debe ser exactamente igual
!PYTHONHASHSEED=0 python src/train.py --seed 42
# Guarda el script en Git con un commit
!git add src/train.py && git commit -q -m "Add seeded, leakage-free training script"
# Muestra solo el último commit
!git log --oneline | head -1

{"seed": 42, "data_md5": "5afc23b9622f8f9ffa60824f4e97d0d5", "accuracy_mean": 0.97367, "accuracy_sd": 0.01859}
{"seed": 42, "data_md5": "5afc23b9622f8f9ffa60824f4e97d0d5", "accuracy_mean": 0.97367, "accuracy_sd": 0.01859}
82b44e9 Add seeded, leakage-free training script


## Step 3 · Record every run (MLflow) · Registra cada ejecución (MLflow)
**EN:** MLflow is a lab notebook for experiments. It saves the seed, the code version, the data fingerprint and the result of **every** run. We try 5 seeds and report the average.  
**ES:** MLflow es un cuaderno de laboratorio para experimentos. Guarda la semilla, la versión del código, la huella de los datos y el resultado de **cada** ejecución. Probamos 5 semillas y reportamos el promedio.

> **🇪🇸 Traducción al español**
>
> ## Paso 3 · Registra cada ejecución (MLflow)
> MLflow es un cuaderno de laboratorio para experimentos. Guarda la semilla, la versión del código, la huella de los datos y el resultado de **cada** ejecución. Probamos 5 semillas y reportamos el promedio.

In [7]:
# MLflow para el registro de experimentos y subprocess para ejecutar comandos de Git
import mlflow, subprocess
# Permite importar el módulo src.train desde la carpeta del proyecto
sys.path.insert(0, "/content/repro-lab")
# Importa la función main del script de entrenamiento
from src.train import main

# Obtiene el hash corto del commit actual de Git (versión exacta del código)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# Guarda los registros de MLflow en una base SQLite dentro del proyecto
mlflow.set_tracking_uri("sqlite:////content/repro-lab/mlflow.db")
# Define (o crea) el experimento
mlflow.set_experiment("session5-repro-lab")

# Ejecuta el entrenamiento con 5 semillas distintas
for seed in [0, 1, 2, 3, 42]:
    # Abre una ejecución (run) de MLflow con nombre "seed-<n>"
    with mlflow.start_run(run_name=f"seed-{seed}"):
        # Entrena y evalúa con esa semilla
        r = main(seed)
        # Registra los parámetros: modelo, semilla, commit de Git y huella de datos
        mlflow.log_params({"model": "logistic_regression", "seed": seed, "git_commit": commit, "data_md5": r["data_md5"]})
        # Registra las métricas: media y DE de la exactitud
        mlflow.log_metrics({"accuracy_mean": r["accuracy_mean"], "accuracy_sd": r["accuracy_sd"]})

# Recupera todas las ejecuciones del experimento como tabla
runs = mlflow.search_runs(experiment_names=["session5-repro-lab"])
# Selecciona las columnas relevantes y ordena por semilla
table = runs[["params.seed", "metrics.accuracy_mean", "metrics.accuracy_sd", "params.git_commit"]].sort_values("params.seed")
# Imprime la tabla sin índice
print(table.to_string(index=False))
# Serie con la exactitud media de cada semilla
acc = runs["metrics.accuracy_mean"]
# Imprime el resultado a reportar: media ± DE entre las 5 semillas
print(f"\nReport this / Reporta esto: {acc.mean():.4f} ± {acc.std(ddof=1):.4f}  (5 seeds / 5 semillas)")

2026/10/10 15:44:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/10 15:44:04 INFO mlflow.store.db.utils: Updating database tables
2026/10/10 15:44:08 INFO mlflow.tracking.fluent: Experiment with name 'session5-repro-lab' does not exist. Creating a new experiment.


params.seed  metrics.accuracy_mean  metrics.accuracy_sd params.git_commit
          0                0.97892              0.01593           82b44e9
          1                0.97893              0.01466           82b44e9
          2                0.97893              0.01466           82b44e9
          3                0.98243              0.00620           82b44e9
         42                0.97367              0.01859           82b44e9

Report this / Reporta esto: 0.9786 ± 0.0031  (5 seeds / 5 semillas)


In [8]:
# Save a small summary table in Git (the MLflow database stays out of Git)
# Guarda una tabla resumen en Git (la base de datos de MLflow queda fuera de Git)
# Exporta la tabla de resultados a CSV sin índice
table.to_csv("results_summary.csv", index=False)
# Agrega el CSV a Git y hace commit
!git add results_summary.csv && git commit -q -m "Add results over 5 seeds"
# Muestra el último commit
!git log --oneline | head -1

8d7d02b Add results over 5 seeds


## Step 4 · A recipe for a stranger · Una receta para un extraño
**EN:** A stranger needs two things: the **exact library versions** (`requirements.txt`) and **clear instructions** (README).  
**ES:** Un extraño necesita dos cosas: las **versiones exactas de las librerías** (`requirements.txt`) e **instrucciones claras** (README).

⚠️ **EN:** Do not use `pip freeze` in Colab: it lists hundreds of Colab-only packages. Pin only what you import.  
**ES:** No uses `pip freeze` en Colab: lista cientos de paquetes de Colab. Fija solo lo que importas.

> **🇪🇸 Traducción al español**
>
> ## Paso 4 · Una receta para un extraño
> Un extraño necesita dos cosas: las **versiones exactas de las librerías** (`requirements.txt`) e **instrucciones claras** (README).
>
> ⚠️ No uses `pip freeze` en Colab: lista cientos de paquetes exclusivos de Colab. Fija solo lo que importas.

In [9]:
# Lee las versiones instaladas de los paquetes y maneja JSON
import importlib.metadata as md, json
# Escribe requirements.txt con las versiones exactas solo de lo que usamos
with open("requirements.txt", "w") as f:
    # Una línea "paquete==versión" por cada dependencia
    for d in ["numpy", "pandas", "scikit-learn"]:
        # Escribe la línea paquete==versión en requirements.txt
        f.write(f"{d}=={md.version(d)}\n")
# Muestra el contenido del archivo
!cat requirements.txt

# Ejecuta el entrenamiento con semilla 42 para obtener el resultado esperado
res = main(42)
# Construye el texto del README (cadena con f-string): instrucciones, salida esperada y entorno
# Contenido del README (cada línea de la cadena siguiente):
#   - Título del proyecto y línea EN: descripción en inglés (regresión logística, datos de Wisconsin)
#   - Línea ES: la misma descripción en español
#   - Sección 'How to reproduce': comandos git clone, pip install -r requirements.txt y python src/train.py --seed 42
#   - Sección 'Expected output': el resultado en JSON calculado arriba ({json.dumps(res)})
#   - Línea con los resultados de 5 semillas (results_summary.csv)
#   - Sección 'Environment': versión de Python, requirements.txt y huella MD5 de los datos
readme = f'''# Reproducible ML Pipeline · Session 5 (UNMSM)

**EN:** Logistic regression on the Wisconsin breast cancer data. Built so a stranger can get the same number.
**ES:** Regresión logística sobre datos de cáncer de mama de Wisconsin. Hecho para que un extraño obtenga el mismo número.

## How to reproduce / Cómo reproducir
```bash
git clone <this-repo-url> && cd repro-lab
pip install -r requirements.txt
PYTHONHASHSEED=0 python src/train.py --seed 42
```

## Expected output / Resultado esperado
```
{json.dumps(res)}
```
Results for 5 seeds / Resultados con 5 semillas: `results_summary.csv`

## Environment / Entorno
Python {platform.python_version()} · packages in `requirements.txt` · data MD5 `{res["data_md5"]}`
'''
# Guarda el README en disco
open("README.md", "w").write(readme)
# Agrega requirements.txt y README a Git y hace commit
!git add requirements.txt README.md && git commit -q -m "Add requirements and README with expected output"
# Muestra el historial completo de commits
!git log --oneline

numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
4809cd8 (HEAD -> main) Add requirements and README with expected output
8d7d02b Add results over 5 seeds
82b44e9 Add seeded, leakage-free training script
191bfc9 Add dataset (breast cancer, 569 rows)
b31198b Start project: README and .gitignore


### Push to GitHub · Subir a GitHub
**EN:** 1) On github.com create an **empty** repository named `repro-lab`. 2) Create a fine-grained token with *Contents: Read and write*. 3) Change `PUSH = False` to `PUSH = True` and run the cell.  
**ES:** 1) En github.com crea un repositorio **vacío** llamado `repro-lab`. 2) Crea un token *fine-grained* con permiso *Contents: Read and write*. 3) Cambia `PUSH = False` a `PUSH = True` y ejecuta la celda.

> **🇪🇸 Traducción al español**
>
> ### Subir a GitHub
> 1) En github.com crea un repositorio **vacío** llamado `repro-lab`. 2) Crea un token *fine-grained* con permiso *Contents: Read and write*. 3) Cambia `PUSH = False` a `PUSH = True` y ejecuta la celda.

In [18]:
# Interruptor de seguridad: no sube nada hasta que lo cambies a True
PUSH = True          # EN: set True when ready / ES: cambia a True cuando estés listo
# Solo si el interruptor está activado
if PUSH:
    # re valida el texto, base64 codifica la credencial, subprocess ejecuta git, os da acceso al entorno
    import re, base64, subprocess, os
    # getpass lee el token sin mostrarlo en pantalla
    from getpass import getpass
    # Pide el nombre de usuario de GitHub (el handle, NO el correo) y quita espacios sobrantes
    user = input("GitHub username (usuario, NO el correo / not your email): ").strip()
    # Valida que sea un usuario válido de GitHub (letras, números y guiones)
    if not re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9-]{0,38})", user):
        # Detiene la celda con un mensaje claro si escribiste un correo u otro texto inválido
        raise SystemExit("❌ Usuario inválido: escribe tu usuario de GitHub (ej. felixmendoza), no tu correo.")
    # Pide el token de forma oculta (nunca se escribe en el código) y quita espacios
    token = getpass("GitHub token (hidden / oculto): ").strip()
    # URL del repositorio SIN credenciales
    repo_url = f"https://github.com/{user}/repro-lab.git"
    # Codifica usuario:token en base64 para la cabecera de autenticación HTTP básica
    auth = base64.b64encode(f"{user}:{token}".encode()).decode()
    # Copia el entorno y le añade la cabecera de autenticación solo para este comando (no queda guardada)
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0",
           "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}"}
    # Borra de memoria el token y su codificación; ya están dentro de env
    del token, auth
    # Carpeta del proyecto creada en el Paso 1
    repo_dir = "/content/repro-lab"
    # Asegura que la rama local se llame main
    subprocess.run(["git", "branch", "-M", "main"], cwd=repo_dir, check=True)
    # Sube main; se captura la salida para mostrar el mensaje real de git sin exponer el token
    r = subprocess.run(["git", "push", repo_url, "main"], cwd=repo_dir, env=env,
                       capture_output=True, text=True)
    # Elimina las credenciales del entorno
    del env
    # Si el push funcionó
    if r.returncode == 0:
        # Confirma y muestra el enlace del repositorio para pegar en Moodle
        print(f"✅ Done / Listo: https://github.com/{user}/repro-lab  -> paste this link in Moodle / pega este enlace en Moodle")
    # Si falló
    else:
        # Muestra el error real de git (no contiene el token)
        print("❌ git push falló / failed:\n", r.stderr)
        # Pista: el repositorio no existe o el token no tiene acceso a él
        if "not found" in r.stderr.lower() or "403" in r.stderr:
            print("➡ Revisa que el repo 'repro-lab' exista en tu cuenta y que el token tenga acceso a él con 'Contents: Read and write'.")
        # Pista: el repositorio remoto ya tiene commits (no está vacío)
        if "rejected" in r.stderr.lower() or "fetch first" in r.stderr.lower():
            print("➡ El repositorio remoto no está vacío: créalo de nuevo SIN README ni .gitignore.")
# Si PUSH es False, solo informa cómo activar la subida
else:
    # Informa cómo activar la subida a GitHub
    print("PUSH is False. Change it to True to upload. / PUSH es False. Cámbialo a True para subir.")

GitHub username (usuario, NO el correo / not your email): felixricharmendoza
GitHub token (hidden / oculto): ··········
✅ Done / Listo: https://github.com/felixricharmendoza/repro-lab  -> paste this link in Moodle / pega este enlace en Moodle


## ✅ Self-check before you submit · Autoevaluación antes de entregar
- [ ] **4 or more commits** with clear messages · **4 o más commits** con mensajes claros
- [ ] **Same seed twice = identical output** · **Misma semilla dos veces = resultado idéntico**
- [ ] **5 seeds logged in MLflow, reported as mean ± SD** · **5 semillas registradas en MLflow, reportadas como media ± DE**
- [ ] **README + requirements.txt** let a stranger get your number · **README + requirements.txt** permiten a un extraño obtener tu número

**Due · Entrega:** GitHub link in Moodle before Session 6 · enlace de GitHub en Moodle antes de la Sesión 6.

### ⭐ Optional extras · Extras opcionales
**EN:** For bigger datasets, version data with [DVC](https://dvc.org/doc). To freeze the whole computer setup, use [Docker](https://docs.docker.com/get-started/).  
**ES:** Para datos grandes, versiona los datos con DVC. Para congelar todo el entorno, usa Docker.

> **🇪🇸 Traducción al español**
>
> ## ✅ Autoevaluación antes de entregar
> - [ ] **4 o más commits** con mensajes claros
> - [ ] **Misma semilla dos veces = resultado idéntico**
> - [ ] **5 semillas registradas en MLflow, reportadas como media ± DE**
> - [ ] **README + requirements.txt** permiten a un extraño obtener tu número
>
> **Entrega:** enlace de GitHub en Moodle antes de la Sesión 6.
>
> ### ⭐ Extras opcionales
> Para datos grandes, versiona los datos con [DVC](https://dvc.org/doc). Para congelar todo el entorno del equipo, usa [Docker](https://docs.docker.com/get-started/).

### **Comprobaciones:**

In [19]:
# Importa subprocess para ejecutar comandos de Git desde Python
import subprocess
# Lista los commits (hash corto y mensaje), del más reciente al más antiguo, dentro del proyecto
log = subprocess.run(["git", "log", "--oneline"], cwd="/content/repro-lab", capture_output=True, text=True).stdout.strip()
# Muestra el historial completo
print(log)
# Cuenta los commits (una línea por commit)
n = len(log.splitlines())
# Informa si se cumple el requisito de 4 o más commits
print(f"\nCommits: {n} ->", "✅ cumple (≥ 4)" if n >= 4 else "❌ faltan commits")

4809cd8 Add requirements and README with expected output
8d7d02b Add results over 5 seeds
82b44e9 Add seeded, leakage-free training script
191bfc9 Add dataset (breast cancer, 569 rows)
b31198b Start project: README and .gitignore

Commits: 5 -> ✅ cumple (≥ 4)


In [20]:
# subprocess ejecuta el script; os da acceso al entorno
import subprocess, os
# Función que ejecuta train.py con una semilla y devuelve su salida (una línea JSON)
def correr(seed):
    # PYTHONHASHSEED=0 se fija al arrancar Python (solo así tiene efecto); cwd = carpeta del proyecto
    r = subprocess.run(["python", "src/train.py", "--seed", str(seed)], cwd="/content/repro-lab",
                       env={**os.environ, "PYTHONHASHSEED": "0"}, capture_output=True, text=True)
    # Devuelve la salida sin saltos de línea sobrantes
    return r.stdout.strip()
# Primera ejecución con la semilla 42
a = correr(42)
# Segunda ejecución con la misma semilla
b = correr(42)
# Muestra ambas salidas
print("Ejecución 1:", a); print("Ejecución 2:", b)
# Comprueba que sean exactamente iguales
print("\nMisma semilla -> idéntico:", "✅ SÍ" if a == b and a else "❌ NO")
# Contraste: con otra semilla el resultado debe cambiar (la semilla sí influye)
print("Otra semilla (7) -> distinto:", "✅ SÍ" if correr(7) != a else "⚠️ igual")

Ejecución 1: {"seed": 42, "data_md5": "5afc23b9622f8f9ffa60824f4e97d0d5", "accuracy_mean": 0.97367, "accuracy_sd": 0.01859}
Ejecución 2: {"seed": 42, "data_md5": "5afc23b9622f8f9ffa60824f4e97d0d5", "accuracy_mean": 0.97367, "accuracy_sd": 0.01859}

Misma semilla -> idéntico: ✅ SÍ
Otra semilla (7) -> distinto: ✅ SÍ


In [21]:
# Importa MLflow y pandas
import mlflow, pandas as pd
# Apunta a la misma base de datos SQLite donde se guardaron las ejecuciones
mlflow.set_tracking_uri("sqlite:////content/repro-lab/mlflow.db")
# Recupera todas las ejecuciones del experimento como tabla
runs = mlflow.search_runs(experiment_names=["session5-repro-lab"])
# Cuenta las semillas distintas registradas
n = runs["params.seed"].nunique()
# Muestra semilla, exactitud media, DE y commit de cada ejecución, ordenadas por semilla
print(runs[["params.seed", "metrics.accuracy_mean", "metrics.accuracy_sd", "params.git_commit"]]
      .sort_values("params.seed").to_string(index=False))
# Exactitud media de cada semilla
acc = runs["metrics.accuracy_mean"]
# Resultado a reportar: media ± desviación estándar muestral (ddof=1) entre semillas
print(f"\nSemillas registradas: {n} ->", "✅ cumple (5)" if n >= 5 else "❌ faltan semillas")
print(f"Reporta esto: {acc.mean():.4f} ± {acc.std(ddof=1):.4f}  (media ± DE, {n} semillas)")

params.seed  metrics.accuracy_mean  metrics.accuracy_sd params.git_commit
          0                0.97892              0.01593           82b44e9
          1                0.97893              0.01466           82b44e9
          2                0.97893              0.01466           82b44e9
          3                0.98243              0.00620           82b44e9
         42                0.97367              0.01859           82b44e9

Semillas registradas: 5 -> ✅ cumple (5)
Reporta esto: 0.9786 ± 0.0031  (media ± DE, 5 semillas)


In [ ]:
# subprocess ejecuta comandos, json lee la salida, re extrae datos, os da acceso al entorno
import subprocess, json, re, os
# Escribe tu usuario de GitHub (el handle, no el correo)
USER = "TU_USUARIO"
# Función auxiliar: ejecuta un comando, falla con mensaje claro si algo sale mal y devuelve la salida
def sh(cmd, cwd=None, env=None):
    # Ejecuta el comando capturando salida y error
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=env, capture_output=True, text=True)
    # Si falló, detiene la celda mostrando el error real
    if r.returncode: raise SystemExit(f"❌ {cmd}\n{r.stderr[-800:]}")
    # Devuelve la salida estándar
    return r.stdout
# Borra cualquier prueba anterior y clona el repositorio en una carpeta nueva (como lo haría un extraño)
sh("rm -rf /tmp/extraño && git clone -q https://github.com/%s/repro-lab.git /tmp/extraño" % USER)
# Crea un entorno virtual limpio, sin los paquetes de Colab
sh("python -m venv /tmp/extraño/.venv")
# Instala SOLO las versiones fijadas en requirements.txt
sh("/tmp/extraño/.venv/bin/pip install -q -r requirements.txt", cwd="/tmp/extraño")
# Ejecuta exactamente el comando del README
salida = sh("PYTHONHASHSEED=0 /tmp/extraño/.venv/bin/python src/train.py --seed 42", cwd="/tmp/extraño").strip()
# Lee del README el resultado esperado (el JSON dentro del bloque de código)
esperado = re.search(r"Expected output.*?```\n(\{.*?\})\n```", open("/tmp/extraño/README.md").read(), re.S).group(1)
# Muestra ambos resultados
print("Obtenido :", salida); print("Esperado :", esperado)
# Compara como JSON para ignorar diferencias de formato
print("\n¿Un extraño obtiene tu número?", "✅ SÍ" if json.loads(salida) == json.loads(esperado) else "❌ NO")